# Notebook 02: Hypothesis Testing and Experiment Evaluation

### Purpose
- Tests whether the Men's and Women's email campaigns drove meaningful changes in customer behavior, and translates statistical results into a business recommendation.

### Objectives
- Determine whether either email campaign had a statistically and practically significant effect on visits, conversions, and revenue.
- Assess robustness of spend results to outliers via winsorization, a necessary step given the heavy-tailed distribution characterized in notebook 01.
- Apply multiple hypothesis testing correction to account for multiple comparisons across 9 tests, and evaluate impact on the conclusions.
- Deliver a concrete recommendation: which campaign to roll out and what incremental revenue it implies per 10,000 customers.

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
from statsmodels.stats import multitest

In [23]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.head(), hillstrom_df.tail())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


## Hypothesis testing for impact metrics (visit, conversion, spend)

In [ ]:
# Visits

comparisons = [("Mens E-Mail", "No E-Mail"), ("Mens E-Mail", "Womens E-Mail"), ("Womens E-Mail", "No E-Mail")]

visit_counts_df = hillstrom_df.groupby("segment")["visit"].agg(["sum", "count"])

results = []
for group1, group2 in comparisons:
    zstat, pvalue = proportions_ztest([visit_counts_df.loc[group1, "sum"], visit_counts_df.loc[group2, "sum"]],
                                      [visit_counts_df.loc[group1, "count"], visit_counts_df.loc[group2, "count"]])
    
    p1 = visit_counts_df.loc[group1, "sum"] / visit_counts_df.loc[group1, "count"]
    p2 = visit_counts_df.loc[group2, "sum"] / visit_counts_df.loc[group2, "count"]
    var1 = p1 * (1 - p1)
    var2 = p2 * (1 - p2)
    n1 = visit_counts_df.loc[group1, "count"]
    n2 = visit_counts_df.loc[group2, "count"]
    se = np.sqrt(var1 / n1 + var2 / n2)
    ci = (p1 - p2) + np.array([-1, 1]) * 1.96 * se
    ci = ci.round(4)

    results.append((group1, group2, zstat, pvalue, p1 - p2, ci))

visit_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "Z-Statistic", "P-Value", "Difference", "95% CI"])
visit_df["Metric"] = "Visit"

display(visit_df)

,Group 1,Group 2,Z-Statistic,P-Value,Difference,95% CI,Metric
0,Mens E-Mail,No E-Mail,22.486041,5.685165e-112,0.076590,"[0.07, 0.0832]",Visit
1,Mens E-Mail,Womens E-Mail,8.684558,3.802165e-18,0.031356,"[0.0243, 0.0384]",Visit
2,Womens E-Mail,No E-Mail,13.949181,3.182403e-44,0.045233,"[0.0389, 0.0516]",Visit


In [ ]:
# Conversions

conversion_counts_df = hillstrom_df.groupby("segment")["conversion"].agg(["sum", "count"])

results = []
for group1, group2 in comparisons:
    zstat, pvalue = proportions_ztest([conversion_counts_df.loc[group1, "sum"], conversion_counts_df.loc[group2, "sum"]],
                                       [conversion_counts_df.loc[group1, "count"], conversion_counts_df.loc[group2, "count"]])
    
    p1 = conversion_counts_df.loc[group1, "sum"] / conversion_counts_df.loc[group1, "count"]
    p2 = conversion_counts_df.loc[group2, "sum"] / conversion_counts_df.loc[group2, "count"]
    n1 = conversion_counts_df.loc[group1, "count"]
    n2 = conversion_counts_df.loc[group2, "count"]
    var1 = p1 * (1 - p1)
    var2 = p2 * (1 - p2)
    se = np.sqrt(var1 / n1 + var2 / n2)
    ci = (p1 - p2) + np.array([-1, 1]) * 1.96 * se
    ci = ci.round(4)

    results.append((group1, group2, zstat, pvalue, p1 - p2, ci))

conversion_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "Z-Statistic", "P-Value", "Difference", "95% CI"])
conversion_df["Metric"] = "Conversion"

display(conversion_df)


,Group 1,Group 2,Z-Statistic,P-Value,Difference,95% CI,Metric
0,Mens E-Mail,No E-Mail,7.385114,1.523224e-13,0.006805,"[0.005, 0.0086]",Conversion
1,Mens E-Mail,Womens E-Mail,3.712584,2.051538e-04,0.003694,"[0.0017, 0.0056]",Conversion
2,Womens E-Mail,No E-Mail,3.779561,1.571051e-04,0.003111,"[0.0015, 0.0047]",Conversion


In [ ]:
# Spend

results = []
for group1, group2 in comparisons:
   spend1 = hillstrom_df.loc[hillstrom_df["segment"] == group1, "spend"]
   spend2 = hillstrom_df.loc[hillstrom_df["segment"] == group2, "spend"]
   tstat, pvalue = ttest_ind(spend1, spend2, equal_var=False)
   
   diff = spend1.mean() - spend2.mean()
   var1 = np.sum((spend1 - spend1.mean()) ** 2) / (len(spend1) - 1)
   var2 = np.sum((spend2 - spend2.mean()) ** 2) / (len(spend2) - 1)
   se = np.sqrt(var1 / len(spend1) + var2 / len(spend2))
   ci = diff + np.array([-1, 1]) * 1.96 * se
   ci = ci.round(4)

   cohens_d = diff / np.sqrt((var1 + var2) / 2)
   
   results.append((group1, group2, tstat, pvalue, diff, ci, cohens_d))
   
spend_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "T-Statistic", "P-Value", "Difference", "95% CI", "Cohen's d"])
spend_df["Metric"] = "Spend"

display(spend_df)


,Group 1,Group 2,T-Statistic,P-Value,Difference,95% CI,Cohen's d,Metric
0,Mens E-Mail,No E-Mail,5.300140,1.163815e-07,0.769827,"[0.4851, 1.0545]",0.051350,Spend
1,Mens E-Mail,Womens E-Mail,2.164018,3.046865e-02,0.345415,"[0.0326, 0.6583]",0.020949,Spend
2,Womens E-Mail,No E-Mail,3.256372,1.129397e-03,0.424412,"[0.169, 0.6799]",0.031512,Spend


### Multiple hypothesis testing correction

In [ ]:
results_df = pd.concat([visit_df, conversion_df, spend_df], ignore_index=True)

results_df["Adjusted P-Value"] = multitest.multipletests(results_df["P-Value"].astype(float), method="fdr_bh")[1]

results_df = results_df[["Metric", "Group 1", "Group 2", "P-Value", "Adjusted P-Value", "Difference", "95% CI", "T-Statistic", "Z-Statistic", "Cohen's d"]]

results_df["P-Value"] = results_df["P-Value"].apply(lambda x: f"{x:.2e}")
results_df["Adjusted P-Value"] = results_df["Adjusted P-Value"].apply(lambda x: f"{x:.2e}")
results_df[["Z-Statistic", "Difference", "Cohen's d"]] = results_df[["Z-Statistic", "Difference", "Cohen's d"]].round(4)

display(results_df)

,Metric,Group 1,Group 2,P-Value,Adjusted P-Value,Difference,95% CI,T-Statistic,Z-Statistic,Cohen's d
0,Visit,Mens E-Mail,No E-Mail,5.69e-112,5.12e-111,0.0766,"[0.07, 0.0832]",NaN,22.4860,NaN
1,Visit,Mens E-Mail,Womens E-Mail,3.80e-18,1.14e-17,0.0314,"[0.0243, 0.0384]",NaN,8.6846,NaN
2,Visit,Womens E-Mail,No E-Mail,3.18e-44,1.43e-43,0.0452,"[0.0389, 0.0516]",NaN,13.9492,NaN
3,Conversion,Mens E-Mail,No E-Mail,1.52e-13,3.43e-13,0.0068,"[0.005, 0.0086]",NaN,7.3851,NaN
4,Conversion,Mens E-Mail,Womens E-Mail,2.05e-04,2.64e-04,0.0037,"[0.0017, 0.0056]",NaN,3.7126,NaN
5,Conversion,Womens E-Mail,No E-Mail,1.57e-04,2.36e-04,0.0031,"[0.0015, 0.0047]",NaN,3.7796,NaN
6,Spend,Mens E-Mail,No E-Mail,1.16e-07,2.09e-07,0.7698,"[0.4851, 1.0545]",5.300140,NaN,0.0514
7,Spend,Mens E-Mail,Womens E-Mail,3.05e-02,3.05e-02,0.3454,"[0.0326, 0.6583]",2.164018,NaN,0.0209
8,Spend,Womens E-Mail,No E-Mail,1.13e-03,1.27e-03,0.4244,"[0.169, 0.6799]",3.256372,NaN,0.0315


### Check impact of outliers via Winsorization

In [ ]:
# Check quantiles for winsorization
hillstrom_df["spend"].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

0.900      0.00000
0.950      0.00000
0.990      0.00000
0.995     68.37135
0.999    243.66049
Name: spend, dtype: float64

In [45]:
cap_value = hillstrom_df["spend"].quantile(0.999)
print(f"Winsorization cap: ${cap_value:.2f}")

results = []
for group1, group2 in comparisons:

    spend1 = hillstrom_df.loc[hillstrom_df["segment"] == group1, "spend"]
    spend1 = np.clip(spend1, 0, cap_value)

    spend2 = hillstrom_df.loc[hillstrom_df["segment"] == group2, "spend"]
    spend2 = np.clip(spend2, 0, cap_value)

    tstat, pvalue = ttest_ind(spend1, spend2, equal_var=False)

    diff = spend1.mean() - spend2.mean()
    var1 = np.sum((spend1 - spend1.mean()) ** 2) / (len(spend1) - 1)
    var2 = np.sum((spend2 - spend2.mean()) ** 2) / (len(spend2) - 1)
    
    se = np.sqrt(var1 / len(spend1) + var2 / len(spend2))
    ci = diff + np.array([-1, 1]) * 1.96 * se
    ci = ci.round(4)

    results.append((group1, group2, tstat, pvalue, diff, ci))

windsorized_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "T-Statistic", "P-Value", "Difference", "95% CI"])
windsorized_df[["T-Statistic", "Difference"]] = windsorized_df[["T-Statistic", "Difference"]].round(4)
windsorized_df["P-Value"] = windsorized_df["P-Value"].apply(lambda x: f"{x:.2e}")
display(windsorized_df)

Winsorization cap: $243.66


,Group 1,Group 2,T-Statistic,P-Value,Difference,95% CI
0,Mens E-Mail,No E-Mail,5.7582,8.57e-09,0.6592,"[0.4348, 0.8835]"
1,Mens E-Mail,Womens E-Mail,2.1672,3.02e-02,0.2762,"[0.0264, 0.5261]"
2,Womens E-Mail,No E-Mail,3.6201,2.95e-04,0.3829,"[0.1756, 0.5902]"


## Business Impact

In [ ]:
# Expected lift per 10,000 customers

impact_df = results_df[(results_df["Group 1"] == "Mens E-Mail") & (results_df["Group 2"] == "No E-Mail")][["Metric", "Difference", "95% CI"]]

display(impact_df)

impact_df[["Difference", "95% CI"]] = impact_df[["Difference", "95% CI"]] * 10000

impact_df

,Metric,Difference,95% CI
0,Visit,0.0766,"[0.07, 0.0832]"
3,Conversion,0.0068,"[0.005, 0.0086]"
6,Spend,0.7698,"[0.4851, 1.0545]"


,Metric,Difference,95% CI
0,Visit,766.0,"[700, 832]"
3,Conversion,68.0,"[50, 86]"
6,Spend,7698.0,"[4851, 10545]"


## Summary

**Results**
- **Men's email performed the best, including versus the women's email**
- Women's email also had statistically significant lift
- Results still significant after multiple testing correction
- The effect sizes are small (Cohen's d = 0.05 for spend), but statistically significant due to the sample size 
- Winsorization had no meaningful impact on results, proceeded with full dataset for analysis

- **Expected lift**
    - The men's email is expected to generate per 10,000 customers:
        - 766 more visits (95% CI [700, 832])
        - 68 more conversions (95% CI [50, 86])
        - $7698 more in spending (95% CI [4851, 10545])